# Global Sensitivity B — n_prior_periods Diagnostics

Train-CV diagnostics only. Test·tuning·threshold 변경은 사용하지 않는다.


## 1. 입력과 26 Feature 계약


In [ ]:
import json, os, sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
ROOT=Path(os.environ.get("KHUDA_PROJECT_ROOT",Path.cwd())).resolve()
while not (ROOT/"code").is_dir():
    if ROOT.parent==ROOT: raise RuntimeError("저장소 안에서 실행하세요.")
    ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
if "code" in sys.modules and not hasattr(sys.modules["code"],"__path__"): del sys.modules["code"]
from code.evaluation.evaluate import calculate_binary_metrics
from code.evaluation.sensitivity_diagnostics import cv_permutation_importance, normalize_oof_columns, paired_sampid_bootstrap_f1, subgroup_oof_metrics
from code.model.locked_sensitivity import load_stage_3_5_locked_params
from code.pipeline.audit import attach_person_period_column, calculate_train_sample_weight, load_selected_feature_names
from code.pipeline.saved_results import load_saved_global_train
RESULT_ROOT=ROOT/"data"/"result"/"baseline_42features"; STAGE35=RESULT_ROOT/"modeling"/"stage_3_5"
DATASET=RESULT_ROOT/"datasets"/"global_dataset.parquet"; SPLIT=RESULT_ROOT/"splits"/"split_ids.csv"; FEATURES=ROOT/"code"/"config"/"features.yaml"; MODELS=ROOT/"code"/"config"/"model_config.yaml"
SELECTED=RESULT_ROOT/"modeling"/"stage_3"/"selected_features.csv"; PARAMS=STAGE35/"final_refined_params.json"; A_SUMMARY=STAGE35/"final_tuning_summary.csv"; A_OOF={m:STAGE35/f"refined_{m}_oof_predictions.parquet" for m in ("logistic_regression","xgboost")}
PERSON=RESULT_ROOT/"datasets"/"person_period.parquet"; LOCK=RESULT_ROOT/"modeling"/"sensitivity_n_prior_periods"; TUNE=LOCK/"tuning"; OUT=LOCK/"diagnostics"
LOCK_SUMMARY=LOCK/"locked_model_summary.csv"; LOCK_OOF={m:LOCK/f"locked_{m}_oof_predictions.parquet" for m in ("logistic_regression","xgboost")}; TUNE_OOF={m:TUNE/f"{m}_oof_predictions.parquet" for m in ("logistic_regression","xgboost")}
required=[DATASET,SPLIT,PERSON,FEATURES,MODELS,SELECTED,PARAMS,A_SUMMARY,*A_OOF.values(),LOCK_SUMMARY,*LOCK_OOF.values(),TUNE/"tuning_summary.csv",TUNE/"best_params.json",*TUNE_OOF.values()]
missing=[str(x) for x in required if not x.exists()]
if missing: raise FileNotFoundError("필요한 Train artifact가 없습니다:\n"+"\n".join(missing))
base=load_saved_global_train(DATASET,SPLIT,FEATURES); selected=load_selected_feature_names(SELECTED); frame=attach_person_period_column(base.to_frame(),PERSON,"n_prior_periods")
assert len(selected)==25 and len(selected)+1==26 and "n_prior_periods" in frame


## 2. n_prior_periods 분포와 subgroup OOF


In [ ]:
def oof(path): return normalize_oof_columns(pd.read_parquet(path))
def rows(strategy, summary, paths, status, stage=None):
    s=pd.read_csv(summary)
    if stage is not None: s=s.loc[s.stage.eq(stage)]
    s=s.set_index("model"); result=[]
    for model,path in paths.items():
        frame=oof(path); metric=calculate_binary_metrics(frame.y_true,frame.y_probability)
        result.append({"strategy":strategy,"model":model,"feature_count":int(s.loc[model,"feature_count"]),"parameter_status":status,"cv_f1_mean":s.loc[model,"cv_f1_mean"],"cv_f1_std":s.loc[model,"cv_f1_std"],"oof_accuracy":metric["accuracy"],"oof_precision":metric["precision"],"oof_recall":metric["recall"],"oof_f1":metric["f1"],"oof_roc_auc":metric["roc_auc"],"oof_average_precision":metric["average_precision"],"predicted_positive_rate":frame.y_predicted.mean()})
    return pd.DataFrame(result)
def paired(labels, frames):
    out=[]
    for label,left,right in labels:
        row=paired_sampid_bootstrap_f1(frames[left],frames[right]); row.insert(0,"comparison",label); out.append(row)
    return pd.concat(out,ignore_index=True)
distribution=(frame.groupby("n_prior_periods").agg(rows=("SAMPID","size"),unique_SAMPID=("SAMPID","nunique"),positive_count=("employment_transition","sum"),positive_rate=("employment_transition","mean")).reset_index())
distribution["ratio"]=distribution.rows/distribution.rows.sum(); display(distribution)
subgroups=[]
for model,path in TUNE_OOF.items():
    value=subgroup_oof_metrics(oof(path),frame.n_prior_periods,group_name="n_prior_periods"); value.insert(0,"model",model); value.insert(0,"strategy","B tuned"); subgroups.append(value)
prior_group_metrics=pd.concat(subgroups,ignore_index=True); display(prior_group_metrics)


## 3. B tuned validation-fold Permutation Importance


In [ ]:
best=json.loads((TUNE/"best_params.json").read_text())
X=frame.loc[:,[*selected,"n_prior_periods"]].reset_index(drop=True)
# 각 fold에서만 재학습하며 Test는 사용하지 않는다.
lr_pi=cv_permutation_importance(X,base.y,base.groups,model_name="logistic_regression",params=best["logistic_regression"],feature_config=FEATURES,model_config=MODELS)
xgb_pi=cv_permutation_importance(X,base.y,base.groups,model_name="xgboost",params=best["xgboost"],feature_config=FEATURES,model_config=MODELS)
display(lr_pi.query("feature == 'n_prior_periods'")); display(xgb_pi.query("feature == 'n_prior_periods'"))


## 4. A / B locked / B tuned 안정성 비교


In [ ]:
def oof(path): return normalize_oof_columns(pd.read_parquet(path))
def rows(strategy, summary, paths, status, stage=None):
    s=pd.read_csv(summary)
    if stage is not None: s=s.loc[s.stage.eq(stage)]
    s=s.set_index("model"); result=[]
    for model,path in paths.items():
        frame=oof(path); metric=calculate_binary_metrics(frame.y_true,frame.y_probability)
        result.append({"strategy":strategy,"model":model,"feature_count":int(s.loc[model,"feature_count"]),"parameter_status":status,"cv_f1_mean":s.loc[model,"cv_f1_mean"],"cv_f1_std":s.loc[model,"cv_f1_std"],"oof_accuracy":metric["accuracy"],"oof_precision":metric["precision"],"oof_recall":metric["recall"],"oof_f1":metric["f1"],"oof_roc_auc":metric["roc_auc"],"oof_average_precision":metric["average_precision"],"predicted_positive_rate":frame.y_predicted.mean()})
    return pd.DataFrame(result)
def paired(labels, frames):
    out=[]
    for label,left,right in labels:
        row=paired_sampid_bootstrap_f1(frames[left],frames[right]); row.insert(0,"comparison",label); out.append(row)
    return pd.concat(out,ignore_index=True)
tuned_summary=pd.read_csv(TUNE/"tuning_summary.csv"); tuned_summary["parameter_status"]="tuned"; tuned_summary["strategy"]="B tuned"
a=rows("A Baseline",A_SUMMARY,A_OOF,"locked",stage="stage_3_5"); locked=rows("B locked",LOCK_SUMMARY,LOCK_OOF,"locked"); tuned=tuned_summary
comparison=pd.concat([a,locked,tuned],ignore_index=True); base_f1=comparison.query("strategy=='A Baseline'").set_index("model").oof_f1; locked_f1=comparison.query("strategy=='B locked'").set_index("model").oof_f1
comparison["delta_oof_f1_vs_A"]=comparison.apply(lambda r:r.oof_f1-base_f1[r.model],axis=1); comparison["locked_to_tuned_delta_f1"]=comparison.apply(lambda r:r.oof_f1-locked_f1[r.model],axis=1); display(comparison)
frames={"A "+m:oof(A_OOF[m]) for m in A_OOF}|{"B locked "+m:oof(LOCK_OOF[m]) for m in LOCK_OOF}|{"B tuned "+m:oof(TUNE_OOF[m]) for m in TUNE_OOF}
bootstrap=pd.concat([paired([( "B locked - A", "B locked "+m,"A "+m),("B tuned - A","B tuned "+m,"A "+m),("B tuned - B locked","B tuned "+m,"B locked "+m)],frames).assign(model=m) for m in ("logistic_regression","xgboost")],ignore_index=True); display(bootstrap)


## 5. 저장·그래프·최종 진단 표


In [ ]:
OUT.mkdir(parents=True,exist_ok=True)
comparison.to_csv(OUT/"diagnostics_summary.csv",index=False); distribution.to_csv(OUT/"n_prior_periods_distribution.csv",index=False); prior_group_metrics.to_csv(OUT/"n_prior_periods_group_metrics.csv",index=False); lr_pi.to_csv(OUT/"logistic_regression_permutation_importance.csv",index=False); xgb_pi.to_csv(OUT/"xgboost_permutation_importance.csv",index=False); bootstrap.to_csv(OUT/"paired_bootstrap_oof.csv",index=False)
fold=pd.read_json(TUNE/"fold_f1.json").melt(var_name="model",value_name="f1").rename_axis("fold").reset_index(); fold.to_csv(OUT/"fold_comparison.csv",index=False)
fig,ax=plt.subplots(); distribution.plot.bar(x="n_prior_periods",y="positive_rate",ax=ax); plt.show()
fig,ax=plt.subplots(); prior_group_metrics.pivot(index="n_prior_periods",columns="model",values="f1").plot.bar(ax=ax); plt.show()
display(comparison); display(pd.concat([lr_pi.query("feature=='n_prior_periods'").assign(model="logistic_regression"),xgb_pi.query("feature=='n_prior_periods'").assign(model="xgboost")])); display(bootstrap)
